# Advanced Time Series Forecasting Libraries

This notebook demonstrates **four advanced libraries** for time series forecasting introduced in the lecture slides, all applied to the same **Beijing PM2.5** dataset:

| Library | Backend | Key Strength |
|---------|---------|-------------|
| **Darts** | PyTorch / sklearn | Unified API for 30+ models, backtesting, ensembles |
| **NeuralProphet** | PyTorch | Extends Facebook Prophet with neural network components |
| **GluonTS** | PyTorch | Probabilistic forecasting, pre-built deep models |
| **PyTorch Forecasting** | PyTorch Lightning | Temporal Fusion Transformer (TFT), interpretable DL |

**Goal:** Same task as the main lab — forecast hourly PM2.5 on the 2014 test set — but using high-level libraries that handle preprocessing, training, and evaluation automatically.

> **Kaggle setup:** Enable **GPU** (Settings → Accelerator → GPU) for faster training.


## 1) Shared Setup & Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

def calc_rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

np.random.seed(42)


In [ ]:
candidates = [
    Path('/kaggle/input/datasets/trongnghia7171/beijing-air-quality/beijing_air_quality.csv'),
    Path('/kaggle/input/beijing-air-quality/beijing_air_quality.csv'),
    Path('data/beijing_air_quality.csv'),
]
raw_path = next((p for p in candidates if p.exists()), None)
if raw_path is None:
    raise FileNotFoundError('Dataset not found. Add the Beijing Air Quality dataset on Kaggle.')

df = pd.read_csv(raw_path, parse_dates=['datetime']).set_index('datetime').sort_index()
df = df[['pm25', 'temperature', 'pressure', 'dewpoint', 'wind_speed']].dropna()

TRAIN_END = '2013-12-31 23:00:00'
train_df = df.loc[:TRAIN_END].copy()
test_df  = df.loc['2014':].copy()

print(f'Full: {df.shape}  Train: {train_df.shape}  Test: {test_df.shape}')
print(f'Train: {train_df.index.min()} → {train_df.index.max()}')
print(f'Test:  {test_df.index.min()} → {test_df.index.max()}')

# We will collect results from all libraries here
all_results = []


---
# 2) Darts

[Darts](https://unit8co.github.io/darts/) provides a **unified API** for 30+ forecasting models — from ARIMA and Prophet to N-BEATS, TFT, and TCN. All share the same `.fit()` / `.predict()` interface, making it trivial to swap models.

**Key features:**
- Built-in backtesting and evaluation utilities
- Automatic scaling and windowing
- Covariates support (past and future)
- Ensemble models and model selection


In [ ]:
!pip install -q darts

# Detect if GPU is usable (sometimes Kaggle's CUDA version mismatches the PyTorch build)
import torch

def get_device():
    """Return 'gpu' if CUDA actually works, else 'cpu'."""
    if torch.cuda.is_available():
        try:
            torch.zeros(1, device='cuda')
            return 'gpu'
        except Exception:
            pass
    return 'cpu'

DEVICE = get_device()
print(f'Using accelerator: {DEVICE}')


In [ ]:
from darts import TimeSeries
from darts.models import NBEATSModel, TCNModel, ExponentialSmoothing
from darts.dataprocessing.transformers import Scaler
from darts.metrics import mae, rmse

target_train = TimeSeries.from_dataframe(train_df, value_cols='pm25', freq='h')
target_test  = TimeSeries.from_dataframe(test_df,  value_cols='pm25', freq='h')

covariates_train = TimeSeries.from_dataframe(
    train_df, value_cols=['temperature', 'pressure', 'dewpoint', 'wind_speed'], freq='h'
)
covariates_full = TimeSeries.from_dataframe(
    df, value_cols=['temperature', 'pressure', 'dewpoint', 'wind_speed'], freq='h'
)

scaler_target = Scaler()
target_train_scaled = scaler_target.fit_transform(target_train)
target_test_scaled  = scaler_target.transform(target_test)

scaler_cov = Scaler()
covariates_full_scaled = scaler_cov.fit_transform(covariates_full)

print('Darts TimeSeries created.')
print(f'  Target train length: {len(target_train)}')
print(f'  Target test length:  {len(target_test)}')


### 2a) Darts — N-BEATS

N-BEATS (Neural Basis Expansion Analysis) is a pure DL architecture designed specifically for time series. It uses stacks of fully-connected layers with residual connections — no RNNs or convolutions needed.


In [ ]:
nbeats = NBEATSModel(
    input_chunk_length=24,
    output_chunk_length=1,
    n_epochs=10,
    batch_size=256,
    random_state=42,
    pl_trainer_kwargs={"accelerator": DEVICE},
)

nbeats.fit(target_train_scaled, verbose=True)

pred_nbeats_scaled = nbeats.predict(n=len(target_test), series=target_train_scaled)
pred_nbeats = scaler_target.inverse_transform(pred_nbeats_scaled)

nbeats_mae = mae(target_test, pred_nbeats)
nbeats_rmse = rmse(target_test, pred_nbeats)

print(f'N-BEATS — MAE: {nbeats_mae:.2f}, RMSE: {nbeats_rmse:.2f}')
all_results.append({'Model': 'Darts N-BEATS', 'MAE': nbeats_mae, 'RMSE': nbeats_rmse, 'Library': 'Darts'})


### 2b) Darts — TCN (Temporal Convolutional Network)

TCN uses dilated causal convolutions — each layer has an exponentially increasing receptive field, allowing it to capture both short and long-term patterns efficiently.


In [ ]:
tcn = TCNModel(
    input_chunk_length=24,
    output_chunk_length=1,
    n_epochs=10,
    batch_size=256,
    random_state=42,
    pl_trainer_kwargs={"accelerator": DEVICE},
)

tcn.fit(target_train_scaled, verbose=True)

pred_tcn_scaled = tcn.predict(n=len(target_test), series=target_train_scaled)
pred_tcn = scaler_target.inverse_transform(pred_tcn_scaled)

tcn_mae = mae(target_test, pred_tcn)
tcn_rmse = rmse(target_test, pred_tcn)

print(f'TCN — MAE: {tcn_mae:.2f}, RMSE: {tcn_rmse:.2f}')
all_results.append({'Model': 'Darts TCN', 'MAE': tcn_mae, 'RMSE': tcn_rmse, 'Library': 'Darts'})


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
target_test[:24*7].plot(label='Actual', ax=ax)
pred_nbeats[:24*7].plot(label='N-BEATS', ax=ax)
pred_tcn[:24*7].plot(label='TCN', ax=ax)
ax.set_title('Darts — First Week of Test Set')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.legend()
plt.tight_layout()
plt.show()


---
# 3) NeuralProphet

[NeuralProphet](https://neuralprophet.com/) extends Facebook's Prophet with **AR-Net** (autoregressive neural network) components. It keeps Prophet's intuitive decomposable model (trend + seasonality + events) but adds:
- Neural network-based autoregression (lagged targets)
- Lagged regressors (covariates)
- Automatic changepoint detection
- PyTorch backend for GPU training

**Best for:** Series with clear trend/seasonality where you also want autoregressive and covariate capabilities.


In [ ]:
!pip install -q neuralprophet


In [ ]:
from neuralprophet import NeuralProphet, set_random_seed

set_random_seed(42)

np_train = train_df.reset_index().rename(columns={'datetime': 'ds', 'pm25': 'y'})
np_test  = test_df.reset_index().rename(columns={'datetime': 'ds', 'pm25': 'y'})

covariate_cols = ['temperature', 'pressure', 'dewpoint', 'wind_speed']

np_train.head()


In [ ]:
m = NeuralProphet(
    n_lags=24,
    n_forecasts=1,
    changepoints_range=0.9,
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=True,
    epochs=10,
    batch_size=256,
    learning_rate=0.01,
    accelerator=DEVICE if DEVICE == 'cpu' else 'auto',
)

for col in covariate_cols:
    m = m.add_lagged_regressor(col, n_lags=24)

train_metrics = m.fit(np_train)
print('Training complete.')


In [ ]:
np_full = pd.concat([np_train, np_test], ignore_index=True)
future = m.make_future_dataframe(np_full, n_historic_predictions=True)
forecast = m.predict(future)

forecast_test = forecast[forecast['ds'] >= '2014-01-01'].copy()

y_true_np = np_test['y'].values[:len(forecast_test)]
y_pred_np = forecast_test['yhat1'].values[:len(y_true_np)]

np_mae = mean_absolute_error(y_true_np, y_pred_np)
np_rmse = calc_rmse(y_true_np, y_pred_np)

print(f'NeuralProphet — MAE: {np_mae:.2f}, RMSE: {np_rmse:.2f}')
all_results.append({'Model': 'NeuralProphet', 'MAE': np_mae, 'RMSE': np_rmse, 'Library': 'NeuralProphet'})


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
week = 24 * 7
ax.plot(y_true_np[:week], label='Actual', color='black', linewidth=2)
ax.plot(y_pred_np[:week], label='NeuralProphet', alpha=0.8, color='darkorange')
ax.set_title('NeuralProphet — First Week of Test Set')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_xlabel('Hour')
ax.legend()
plt.tight_layout()
plt.show()


### NeuralProphet — Component Decomposition

One of NeuralProphet's strengths is **built-in decomposition** — you can visualize trend, seasonality, and AR contributions separately.


In [ ]:
m.plot_components(forecast.tail(24 * 30))
plt.tight_layout()
plt.show()


---
# 4) GluonTS

[GluonTS](https://ts.gluon.ai/) (by Amazon/AWS) specializes in **probabilistic forecasting** — models output full prediction distributions (quantiles, confidence intervals), not just point estimates.

**Key features:**
- DeepAR, Transformer, SimpleFeedForward, and many more
- Automatic dataset creation from pandas
- Built-in evaluation with multiple metrics
- Designed for production-scale forecasting

**Best for:** When you need prediction intervals, uncertainty quantification, or are working at scale.


In [ ]:
!pip install -q gluonts[torch] lightning


In [ ]:
from gluonts.dataset.pandas import PandasDataset
from gluonts.torch.model.deepar import DeepAREstimator
from gluonts.torch.model.simple_feedforward import SimpleFeedForwardEstimator
from gluonts.evaluation import make_evaluation_predictions, Evaluator

PREDICTION_LENGTH = len(test_df)

gluon_full = df[['pm25']].copy()
gluon_full.index.freq = 'h'

gluon_dataset = PandasDataset.from_long_dataframe(
    gluon_full.reset_index().rename(columns={'datetime': 'timestamp'}),
    target='pm25',
    timestamp='timestamp',
)

print(f'GluonTS dataset created. Prediction length: {PREDICTION_LENGTH}')


### 4a) GluonTS — DeepAR

DeepAR is an autoregressive recurrent network that produces probabilistic forecasts. It models the conditional distribution at each time step using a parametric likelihood (e.g., Gaussian, Student's t).


In [ ]:
deepar_estimator = DeepAREstimator(
    prediction_length=24 * 7,
    context_length=24,
    freq='h',
    trainer_kwargs={"max_epochs": 10, "accelerator": DEVICE},
)

deepar_predictor = deepar_estimator.train(training_data=gluon_dataset)
print('DeepAR training complete.')


In [ ]:
forecast_it, ts_it = make_evaluation_predictions(
    dataset=gluon_dataset,
    predictor=deepar_predictor,
    num_samples=100,
)

forecasts = list(forecast_it)
tss = list(ts_it)

evaluator = Evaluator(quantiles=[0.1, 0.5, 0.9])
agg_metrics, item_metrics = evaluator(tss, forecasts)

deepar_rmse = np.sqrt(agg_metrics.get("MSE", np.nan))
deepar_mase = agg_metrics.get("MASE", np.nan)

# Compute MAE from the median forecast vs actuals
if forecasts:
    fc = forecasts[0]
    fc_median = fc.quantile(0.5)
    actual_tail = tss[0].values[-len(fc_median):].flatten()
    deepar_mae = mean_absolute_error(actual_tail, fc_median)
else:
    deepar_mae = np.nan

print(f"DeepAR — MAE: {deepar_mae:.2f}, RMSE: {deepar_rmse:.2f}, MASE: {deepar_mase:.4f}")
all_results.append({"Model": "DeepAR (GluonTS)", "MAE": deepar_mae, "RMSE": deepar_rmse, "Library": "GluonTS"})

print(f"
Key aggregated metrics:")
for k in ["MASE", "MSE", "RMSE", "sMAPE", "mean_wQuantileLoss"]:
    v = agg_metrics.get(k, None)
    if v is not None:
        print(f"  {k}: {v:.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

if forecasts:
    fc = forecasts[0]
    fc_median = fc.quantile(0.5)
    fc_low = fc.quantile(0.1)
    fc_high = fc.quantile(0.9)
    fc_index = fc.index.to_timestamp() if hasattr(fc.index, 'to_timestamp') else fc.index

    actual_slice = gluon_full['pm25'].iloc[-(len(fc_median) + 24*7):]
    ax.plot(actual_slice.index, actual_slice.values, color='black', linewidth=1.5, label='Actual')
    ax.plot(fc_index, fc_median, color='dodgerblue', label='DeepAR Median')
    ax.fill_between(fc_index, fc_low, fc_high, alpha=0.2, color='dodgerblue', label='80% PI')

ax.set_title('GluonTS DeepAR — Probabilistic Forecast')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.legend()
plt.tight_layout()
plt.show()


> **Note:** GluonTS is designed for probabilistic evaluation (MASE, quantile losses, coverage) rather than simple MAE/RMSE. The shaded prediction intervals are a key advantage over point-forecast-only models.


---
# 5) PyTorch Forecasting

[PyTorch Forecasting](https://pytorch-forecasting.readthedocs.io/) builds on PyTorch Lightning and provides:

- **Temporal Fusion Transformer (TFT)** — state-of-the-art interpretable multi-horizon forecasting
- **N-HiTS** — efficient variant of N-BEATS for long horizons
- Built-in variable selection, attention visualization, and interpretability tools
- `TimeSeriesDataSet` handles all windowing and feature encoding automatically

**Best for:** When you want both high accuracy AND interpretability (variable importance, attention weights).


In [ ]:
!pip install -q pytorch-forecasting pytorch-lightning


In [ ]:
import pytorch_lightning as pl
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import MAE

pl.seed_everything(42)

ptf_df = df.reset_index().copy()
ptf_df['time_idx'] = np.arange(len(ptf_df))
ptf_df['group'] = 'beijing'

ptf_df['hour'] = ptf_df['datetime'].dt.hour.astype(str)
ptf_df['month'] = ptf_df['datetime'].dt.month.astype(str)
ptf_df['day_of_week'] = ptf_df['datetime'].dt.dayofweek.astype(str)

train_cutoff = ptf_df.loc[ptf_df['datetime'] < '2014', 'time_idx'].max()

print(f'Total rows: {len(ptf_df)}, Train cutoff time_idx: {train_cutoff}')
ptf_df.head()


In [ ]:
MAX_ENCODER_LENGTH = 24
MAX_PREDICTION_LENGTH = 6

training = TimeSeriesDataSet(
    ptf_df[ptf_df['time_idx'] <= train_cutoff],
    time_idx='time_idx',
    target='pm25',
    group_ids=['group'],
    max_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH,
    time_varying_known_reals=['time_idx'],
    time_varying_known_categoricals=['hour', 'month', 'day_of_week'],
    time_varying_unknown_reals=['pm25', 'temperature', 'pressure', 'dewpoint', 'wind_speed'],
    target_normalizer=GroupNormalizer(groups=['group']),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

validation = TimeSeriesDataSet.from_dataset(
    training, ptf_df, min_prediction_idx=train_cutoff + 1
)

train_dl = training.to_dataloader(train=True, batch_size=256, num_workers=0)
val_dl   = validation.to_dataloader(train=False, batch_size=256, num_workers=0)

print(f'Training samples: {len(training)}, Validation samples: {len(validation)}')


### 5a) Temporal Fusion Transformer (TFT)

TFT combines:
- **Variable Selection Networks** — learns which features matter at each time step
- **Multi-head Attention** — captures long-range temporal dependencies
- **Gating mechanisms** — suppresses unnecessary components

It produces both point forecasts and prediction intervals, plus interpretability outputs.


In [ ]:
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.01,
    hidden_size=32,
    attention_head_size=2,
    dropout=0.1,
    hidden_continuous_size=16,
    loss=MAE(),
    log_interval=10,
    reduce_on_plateau_patience=3,
)

print(f'TFT parameters: {tft.size()/1e3:.1f}k')


In [ ]:
trainer = pl.Trainer(
    max_epochs=10,
    accelerator=DEVICE,
    gradient_clip_val=0.1,
    enable_progress_bar=True,
    enable_model_summary=True,
)

trainer.fit(tft, train_dataloaders=train_dl, val_dataloaders=val_dl)
print('TFT training complete.')


In [ ]:
import torch

predictions = tft.predict(val_dl, return_x=True)

pred_values = predictions.output.cpu().numpy()
actuals = torch.cat([y[0] for x, y in iter(val_dl)]).cpu().numpy()

min_len = min(len(actuals.flatten()), len(pred_values.flatten()))
act_flat = actuals.flatten()[:min_len]
pred_flat = pred_values.flatten()[:min_len]


tft_mae = mean_absolute_error(act_flat, pred_flat)
tft_rmse = calc_rmse(act_flat, pred_flat)

print(f'TFT — MAE: {tft_mae:.2f}, RMSE: {tft_rmse:.2f}')
all_results.append({'Model': 'TFT (PyTorch Forecasting)', 'MAE': tft_mae, 'RMSE': tft_rmse, 'Library': 'PyTorch Forecasting'})


### TFT — Interpretability

One of TFT's key advantages: built-in variable importance and attention visualizations.


In [ ]:
try:
    interpretation = tft.interpret_output(predictions.output, reduction="sum")
    tft.plot_interpretation(interpretation)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Interpretation plot not available: {e}')
    print('This is common when the validation set is small or the model has few epochs.')


---
# 6) Grand Comparison


In [ ]:
results_df = pd.DataFrame(all_results).set_index('Model').sort_values('RMSE')
results_df.style.format({'MAE': '{:.2f}', 'RMSE': '{:.2f}'}).highlight_min(
    subset=['MAE', 'RMSE'], axis=0, color='#d4edda'
)


In [ ]:
colors = {
    'Darts': '#3498db',
    'NeuralProphet': '#e67e22',
    'GluonTS': '#2ecc71',
    'PyTorch Forecasting': '#9b59b6',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
comp = results_df.reset_index()

for metric, ax in zip(['MAE', 'RMSE'], axes):
    bars = ax.barh(
        comp['Model'], comp[metric],
        color=[colors.get(lib, 'gray') for lib in comp['Library']]
    )
    ax.set_title(metric)
    ax.set_xlabel(metric)

handles = [plt.Rectangle((0,0),1,1, color=c) for c in colors.values()]
fig.legend(handles, colors.keys(), loc='upper right', fontsize=9, title='Library')
plt.tight_layout()
plt.show()


## 7) Library Comparison & When to Use Each

### Quick Reference

| Library | Best For | Ease of Use | Interpretability | Probabilistic |
|---------|---------|-------------|-----------------|---------------|
| **Darts** | Rapid prototyping, model comparison | ⭐⭐⭐⭐⭐ | Medium | Yes |
| **NeuralProphet** | Series with trend + seasonality | ⭐⭐⭐⭐ | High (decomposition) | Limited |
| **GluonTS** | Probabilistic forecasting, production | ⭐⭐⭐ | Low | ⭐⭐⭐⭐⭐ |
| **PyTorch Forecasting** | Interpretable DL, TFT | ⭐⭐⭐ | High (attention, var. importance) | Yes |

### Recommendations

1. **Start with Darts** if you want to quickly compare many models with minimal code changes.
2. **Use NeuralProphet** when your data has clear trend/seasonality and you want intuitive decomposition.
3. **Use GluonTS** when prediction intervals and uncertainty quantification are critical (risk management, capacity planning).
4. **Use PyTorch Forecasting (TFT)** when you need both high accuracy and the ability to explain *which features* and *which time steps* drove the forecast.

### vs. Manual Keras/TF Implementation

The manual LSTM/CNN from the main lab gives you full control but requires writing all preprocessing, windowing, and evaluation code yourself. These libraries handle that automatically, letting you focus on model selection and interpretation. The trade-off is flexibility vs. convenience.
